In [0]:
import json
import random
import string
import uuid
import time
from datetime import datetime, timedelta, timezone
from pyspark.sql.types import StringType, StructField, StringType, StructType
from pyspark.sql.functions import col, lit, current_timestamp

# ── 0. Initial data (kept as-is) ─────────────────────────────────────────────
initial_data = [
    {"DeviceId": "4fc82b26aecb47d2", "TimestampUTC": "2023-01-01T07:11:29", "DeviceType": "Washing machine", "Status": "Program Start", "Attributes": {"Temperature": 60, "Temperature_Unit": "Celsius", "SpinningSpeed": 900, "SpinningSpeed_Unit": "Rotations per Minute", "TwinDos": [90, 100], "TwinDos_Unit": "Percentage"}},
    {"DeviceId": "4fc82b26aecb47d2", "TimestampUTC": "2023-01-01T09:27:12", "DeviceType": "Washing machine", "Status": "Program End", "Attributes": {"Temperature": 60, "Temperature_Unit": "Celsius", "SpinningSpeed": 900, "SpinningSpeed_Unit": "Rotations per Minute", "TwinDos": [90, 95], "TwinDos_Unit": "Percentage"}},
    {"DeviceId": "6b51d431df5d7f14", "TimestampUTC": "2023-01-01T08:12:00", "DeviceType": "Washing machine", "Status": "Program Start", "Attributes": {"Temperature": 104, "Temperature_Unit": "Fahrenheit", "SpinningSpeed": 1100, "SpinningSpeed_Unit": "Rotations per Minute"}},
    {"DeviceId": "6b51d431df5d7f14", "TimestampUTC": "2023-01-01T08:12:42", "DeviceType": "Washing machine", "Status": "Program Failure", "Attributes": {"FailureIds": [10, 17, 23]}},
    {"DeviceId": "3fdba35f04dc8c46", "TimestampUTC": "2023-01-01T21:57:01", "DeviceType": "Washing machine", "Status": "Program Failure", "Attributes": {"FailureIds": [17]}},
    {"DeviceId": "eb1e33e8a81b697b", "TimestampUTC": "2023-01-01T12:23:34", "DeviceType": "Coffee Machine", "Status": "Program End", "Attributes": [{"Id": "Grinding", "Value": 100, "Unit": "Percentage"}, {"Id": "Temperature", "Value": 86, "Unit": "Celsius"}]},
    {"DeviceId": "e29c9c180c6279b0", "TimestampUTC": "2023-01-01T12:23:34", "DeviceType": "Coffee Machine", "Status": "Program End", "Attributes": [{"Id": "Temperature", "Value": 86, "Unit": "Celsius"}]},
    {"DeviceId": "d59eced1ded07f84", "TimestampUTC": "2023-01-01T18:37:37", "DeviceType": "Dishwasher", "Status": "Program End", "Attributes": {"Temperature": 40, "Temperature_Unit": "Fahrenheit", "QuickPowerWashActive": 1, "QuickPowerWashActive_Unit": "Bit"}}
]

# ── 1. Synthetic data generators — one per device type ───────────────────────

def random_suffix(n=13):
    return ''.join(random.choices(string.ascii_lowercase + string.digits, k=n))

def random_timestamp(start="2023-01-01", days=365):
    base = datetime.strptime(start, "%Y-%m-%d")
    return (base + timedelta(
        days=random.randint(0, days),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )).strftime("%Y-%m-%dT%H:%M:%S")

def gen_washing_machine(device_id):
    """Mirrors the real washing machine message patterns."""
    """weights reflects real device telemetry distributions, failures are rare, which matters for downstream anomaly detection logic"""
    status = random.choices(
        ["Program Start", "Program End", "Program Failure"],
        weights=[0.40, 0.45, 0.15]
    )[0]

    if status == "Program Failure":
        failure_pool = [10, 17, 23, 31, 42, 55]
        attrs = {"FailureIds": random.sample(failure_pool, k=random.randint(1, 3))}
    else:
        unit = random.choice(["Celsius", "Fahrenheit"])
        temp = random.randint(30, 95) if unit == "Celsius" else random.randint(86, 203)
        attrs = {
            "Temperature": temp,
            "Temperature_Unit": unit,
            "SpinningSpeed": random.choice([600, 800, 900, 1000, 1100, 1200, 1400]),
            "SpinningSpeed_Unit": "Rotations per Minute",
        }
        if random.random() > 0.4:           # TwinDos is optional
            attrs["TwinDos"] = [random.randint(70, 100), random.randint(70, 100)]
            attrs["TwinDos_Unit"] = "Percentage"

    return {"DeviceId": device_id, "TimestampUTC": random_timestamp(),
            "DeviceType": "Washing machine", "Status": status, "Attributes": attrs}

def gen_coffee_machine(device_id):
    """Mirrors the real coffee machine message pattern (list-style Attributes)."""
    status = random.choices(
        ["Program Start", "Program End", "Program Failure"],
        weights=[0.35, 0.55, 0.10]
    )[0]

    attr_pool = [
        {"Id": "Grinding",     "Value": random.randint(60, 100), "Unit": "Percentage"},
        {"Id": "Temperature",  "Value": random.randint(80, 96),  "Unit": "Celsius"},
        {"Id": "WaterPressure","Value": round(random.uniform(8, 10), 1), "Unit": "Bar"},
        {"Id": "CupSize",      "Value": random.choice([30, 60, 120, 240]), "Unit": "Millilitre"},
    ]
    # Always include Temperature; optionally add 1–2 other attributes
    attrs = [attr_pool[1]] + random.sample([attr_pool[0], attr_pool[2], attr_pool[3]],
                                            k=random.randint(0, 2))

    return {"DeviceId": device_id, "TimestampUTC": random_timestamp(),
            "DeviceType": "Coffee Machine", "Status": status, "Attributes": attrs}

def gen_dishwasher(device_id):
    """Mirrors the real dishwasher message pattern."""
    status = random.choices(
        ["Program Start", "Program End", "Program Failure"],
        weights=[0.38, 0.50, 0.12]
    )[0]

    if status == "Program Failure":
        failure_pool = [5, 11, 14, 22, 33]
        attrs = {"FailureIds": random.sample(failure_pool, k=random.randint(1, 2))}
    else:
        unit = random.choice(["Celsius", "Fahrenheit"])
        temp = random.randint(40, 75) if unit == "Celsius" else random.randint(104, 167)
        attrs = {
            "Temperature": temp,
            "Temperature_Unit": unit,
            "QuickPowerWashActive": random.randint(0, 1),
            "QuickPowerWashActive_Unit": "Bit",
        }

    return {"DeviceId": device_id, "TimestampUTC": random_timestamp(),
            "DeviceType": "Dishwasher", "Status": status, "Attributes": attrs}

# ── 2. Generate a pool of synthetic device IDs ───────────────────────────────
#       Scale NUM_DEVICES to taste — 100_000 simulates a global fleet
NUM_DEVICES       = 1_000   # ← was 100_000; use 1k for fast demo
EVENTS_PER_DEVICE = 3       # ← was 5; fewer events per device
WRITE_BATCH_SIZE  = 2_000   # ← was 5_000; smaller batches = less pressure

#NUM_DEVICES   = 100_000
#EVENTS_PER_DEVICE = 5       # avg events per device → total ~500k rows

device_type_weights = {"Washing machine": 0.50, "Coffee Machine": 0.30, "Dishwasher": 0.20}
generators = {
    "Washing machine": gen_washing_machine,
    "Coffee Machine":  gen_coffee_machine,
    "Dishwasher":      gen_dishwasher,
}

device_registry = {
    f"syn_{random_suffix(13)}": random.choices(
        list(device_type_weights.keys()),
        weights=list(device_type_weights.values())
    )[0]
    for _ in range(NUM_DEVICES)
}

print(f"Device registry built: {len(device_registry):,} synthetic devices")
print("Type breakdown:",
      {t: sum(1 for v in device_registry.values() if v == t)
       for t in device_type_weights})

# ── 3. Generate synthetic events ─────────────────────────────────────────────
synthetic_data = [
    generators[dtype](did)
    for did, dtype in device_registry.items()
    for _ in range(random.randint(1, EVENTS_PER_DEVICE * 2))   # variable event count
]

print(f"Synthetic events generated: {len(synthetic_data):,}")

# ── 4. Combine with initial data ─────────────────────────────────────────────
all_data = initial_data + synthetic_data
print(f"Total events (initial + synthetic): {len(all_data):,}")

# ── 4b. Inject intentionally malformed records for dead-letter demo ───────────
corrupt_data = [
    # 1. Non-serializable Python object (datetime is not JSON-serializable by default)
    {"DeviceId": "bad_001", "TimestampUTC": datetime.now(), "DeviceType": "Washing machine", "Status": "Program Start", "Attributes": {}},

    # 2. Circular reference — json.dumps will raise ValueError
    (lambda d: d.update({"self": d}) or d)({"DeviceId": "bad_002", "TimestampUTC": "2023-01-01T00:00:00", "DeviceType": "Dishwasher", "Status": "Program End", "Attributes": {}}),

    # 3. Contains a non-serializable custom object
    {"DeviceId": "bad_003", "TimestampUTC": "2023-01-01T00:00:00", "DeviceType": "Coffee Machine", "Status": "Program End", "Attributes": {"Temperature": object()}},
]

all_data = initial_data + synthetic_data + corrupt_data + exact_duplicates
print(f"Total events (initial + synthetic + corrupt): {len(all_data):,}")
print(f"Corrupt records injected: {len(corrupt_data)} (expect {len(corrupt_data)} dead-letter rows)")

# ── 4c. Inject intentional duplicates for silver deduplication demo ───────────

# Strategy 1 — exact duplicates: same record copied as-is
exact_duplicates = random.sample(initial_data, k=3)

# ── 5. Write to Delta in bulk batches, then read back as versioned micro-batches

BRONZE_SCHEMA = StructType([
    StructField("value",        StringType()),
    StructField("_ingested_at", StringType()),
    StructField("_source_file", StringType()),
])

demo_table = "device_stream_demo"
spark.sql(f"DROP TABLE IF EXISTS {demo_table}")
spark.createDataFrame([], BRONZE_SCHEMA) \
     .write.format("delta").saveAsTable(demo_table)
#spark.createDataFrame([], StringType()).toDF("value").write.format("delta").saveAsTable(demo_table)

WRITE_BATCH_SIZE = 5_000    # rows per Delta append (tune to cluster memory)
write_batches = [all_data[i:i+WRITE_BATCH_SIZE]
                 for i in range(0, len(all_data), WRITE_BATCH_SIZE)]

# ── 6. Replay as simulated Kafka micro-batches using Delta versioning ─────────
MAX_BATCH_RETRIES = 3

DEAD_LETTER_TABLE = "device_stream_demo_dead_letter"

def safe_serialize(record):
    try:
        value = json.dumps(record)
        return (value, None)           # (good_value, error)
    except (TypeError, ValueError) as e:
        dead = json.dumps({"_error": str(e), "_raw": str(record)})
        return (None, dead)            # (good_value, error)

spark.sql(f"DROP TABLE IF EXISTS {DEAD_LETTER_TABLE}")
spark.createDataFrame([], StructType([
    StructField("_raw_error",   StringType()),
    StructField("_ingested_at", StringType()),
])).write.format("delta").saveAsTable(DEAD_LETTER_TABLE)

for i, batch in enumerate(write_batches, start=1):
    #rows = [(json.dumps(r),) for r in batch]
    rows = [(safe_serialize(r),) for r in batch]
    ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")  # one timestamp per batch, same as notebook's bronze
    
    serialized   = [safe_serialize(r) for r in batch]
    good_rows    = [(val,) for val, err in serialized if err is None]
    bad_rows     = [(err,) for val, err in serialized if err is not None]
    
    if bad_rows:
        print(f"  Batch {i}: {len(bad_rows)} bad record(s) -> dead-letter table")
        spark.createDataFrame(bad_rows, ["_raw_error"]) \
             .withColumn("_ingested_at", lit(ingested_at)) \
             .write.format("delta").mode("append").saveAsTable(DEAD_LETTER_TABLE)

    if not good_rows:
        print(f"  Batch {i}: no valid records, skipping main write")
        continue

    rows = good_rows   # ← replaces the old: rows = [(json.dumps(r),) for r in batch]

    for attempt in range(1, MAX_BATCH_RETRIES + 1):
        try:
            spark.createDataFrame(rows, ["value"]) \
                 .withColumn("_ingested_at", lit(ingested_at)) \
                 .withColumn("_source_file", lit("kafka://device_stream_demo")) \
                 .write.format("delta").mode("append").saveAsTable(demo_table)
            print(f"  Written write-batch {i}/{len(write_batches)} ({len(good_rows):,} rows)")
            break
        except Exception as e:
            print(f"  Batch {i} attempt {attempt} failed: {e}")
            if attempt < MAX_BATCH_RETRIES:
                time.sleep(5 * attempt)
            else:
                raise
    time.sleep(0.5)

print(f"\nAll data written to '{demo_table}'.")

# ── 6. Replay as simulated Kafka micro-batches using Delta versioning ─────────
history = spark.sql(f"DESCRIBE HISTORY {demo_table}") \
               .select("version").orderBy("version").collect()
versions = [row["version"] for row in history]

print(f"\nReplaying {len(versions)} Delta versions as simulated Kafka batches...\n")

MAX_DISPLAY_BATCHES = 10    # ← set to None to replay all versions

versions_to_replay = versions[:MAX_DISPLAY_BATCHES] if MAX_DISPLAY_BATCHES else versions
print(f"Replaying {len(versions_to_replay)} of {len(versions)} Delta versions...\n")

for version in versions_to_replay:
    batch_df = spark.read.format("delta") \
                    .option("versionAsOf", version) \
                    .table(demo_table)
    count = batch_df.count()
    print(f"{'='*65}")
    print(f"  Kafka batch ~ Delta version {version} | {count:,} cumulative rows")
    print(f"{'='*65}")

    print("  Sample — initial records (up to 2):")
    batch_df.filter(~col("value").contains('"syn_')) \
            .limit(2).show(truncate=80)

    print("  Sample — synthetic records (up to 2):")
    batch_df.filter(col("value").contains('"syn_')) \
            .limit(2).show(truncate=80)

print("\nDLQ (if any):")
spark.read.format("delta").table(DEAD_LETTER_TABLE).show(truncate=False)
print("\nSimulation complete.")

Device registry built: 1,000 synthetic devices
Type breakdown: {'Washing machine': 488, 'Coffee Machine': 294, 'Dishwasher': 218}
Synthetic events generated: 3,416
Total events (initial + synthetic): 3,424
Total events (initial + synthetic + corrupt): 3,427
Corrupt records injected: 3 (expect 3 dead-letter rows)
  Batch 1: 3 bad record(s) -> dead-letter table
  Written write-batch 1/1 (3,424 rows)

All data written to 'device_stream_demo'.

Replaying 2 Delta versions as simulated Kafka batches...

Replaying 2 of 2 Delta versions...

  Kafka batch ~ Delta version 0 | 0 cumulative rows
  Sample — initial records (up to 2):
+-----+------------+------------+
|value|_ingested_at|_source_file|
+-----+------------+------------+
+-----+------------+------------+

  Sample — synthetic records (up to 2):
+-----+------------+------------+
|value|_ingested_at|_source_file|
+-----+------------+------------+
+-----+------------+------------+

  Kafka batch ~ Delta version 1 | 3,424 cumulative rows
 

In [0]:
%pip install rapidfuzz recordlinkage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.9/926.9 kB 38.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import pandas as pd
import recordlinkage
from rapidfuzz import fuzz
from pyspark.sql.functions import (
    col, from_json, lit, trim, lower,
    to_timestamp, to_utc_timestamp, date_format,
    regexp_replace, current_timestamp
)
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

# ── Configuration ─────────────────────────────────────────────────────────────
SILVER_TABLE                  = "device_stream_silver"
SILVER_DEAD_LETTER_TABLE      = "device_stream_silver_dead_letter"
FUZZY_SIMILARITY_THRESHOLD    = 90   # RapidFuzz score 0-100; >= 90 treated as duplicate
ALERT_INVALID_THRESHOLD_PCT   = 5.0
ALERT_POST_DEDUP_THRESHOLD_PCT = 20.0

DEDUP_KEY = ["DeviceId", "TimestampUTC", "DeviceType", "Status"]

ID_SCHEMA = StructType([
    StructField("DeviceId",     StringType()),
    StructField("TimestampUTC", StringType()),
    StructField("DeviceType",   StringType()),
    StructField("Status",       StringType()),
    StructField("Attributes",   StringType()),
])

silver_processed_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# ── 1. Read bronze ─────────────────────────────────────────────────────────────
bronze_df = spark.read.format("delta").table("device_stream_demo")
n_bronze = bronze_df.count()
print(f"Silver | Bronze rows read        : {n_bronze:,}")

# ── 2. Exclude dead-letter records that arrived via bronze ─────────────────────
# Dead-letter records are corrupt JSON rows safe_serialize() flagged with "_error".
# They have no valid identity fields so exclude them before any silver processing.
valid_bronze = bronze_df.filter(~col("value").contains('"_error"'))
n_valid_bronze = valid_bronze.count()
print(f"Silver | Valid bronze rows        : {n_valid_bronze:,}")
print(f"Silver | Corrupt rows excluded    : {n_bronze - n_valid_bronze:,}")

# ── 3. Parse JSON value column into identity + Attributes columns ──────────────
parsed_df = (
    valid_bronze
    .withColumn("parsed", from_json(col("value"), ID_SCHEMA))
    .select(
        col("parsed.DeviceId").alias("DeviceId"),
        col("parsed.TimestampUTC").alias("TimestampUTC"),
        col("parsed.DeviceType").alias("DeviceType"),
        col("parsed.Status").alias("Status"),
        col("parsed.Attributes").alias("Attributes"),
        col("_ingested_at"),
        col("_source_file"),
    )
)

# ── 4. Validate required fields ────────────────────────────────────────────────
REQUIRED = ["DeviceId", "TimestampUTC", "DeviceType", "Status"]
valid_filter = None
for c in REQUIRED:
    cond = col(c).isNotNull() & (trim(col(c)) != "")
    valid_filter = cond if valid_filter is None else valid_filter & cond

valid_df   = parsed_df.filter(valid_filter)
invalid_df = parsed_df.filter(~valid_filter)

n_valid   = valid_df.count()
n_invalid = invalid_df.count()
print(f"Silver | Rows passing validation  : {n_valid:,}")
print(f"Silver | Rows failing validation  : {n_invalid:,}")

# Route invalid rows to silver dead-letter table
spark.sql(f"DROP TABLE IF EXISTS {SILVER_DEAD_LETTER_TABLE}")
invalid_df \
    .withColumn("_silver_processed_at", lit(silver_processed_at)) \
    .withColumn("_rejection_reason", lit("missing_required_field")) \
    .write.format("delta").mode("overwrite").saveAsTable(SILVER_DEAD_LETTER_TABLE)

# ── 5. Normalize DeviceType casing ────────────────────────────────────────────
# Real devices often send inconsistent casing e.g. "washing machine" vs "Washing machine"
# Normalize to title case so downstream gold filters work reliably
normalized_df = valid_df.withColumn(
    "DeviceType",
    # Title-case each word: "washing machine" -> "Washing Machine"
    # Spark has no native initcap-per-word on multi-word strings so use initcap()
    # which does exactly this (uppercases first letter after any whitespace/special char)
    col("DeviceType")  # keep as-is here; initcap shown below for reference
    # F.initcap(col("DeviceType"))  # ← uncomment to enforce title case
)

# ── 6. Normalize timestamps ────────────────────────────────────────────────────
normalized_df = (
    normalized_df
    .withColumn(
        "TimestampUTC",
        date_format(
            to_utc_timestamp(
                to_timestamp(
                    regexp_replace(col("TimestampUTC"), "Z$", ""),
                    "yyyy-MM-dd'T'HH:mm:ss",
                ),
                "UTC",
            ),
            "yyyy-MM-dd'T'HH:mm:ss'Z'",
        )
    )
    .filter(col("TimestampUTC").isNotNull())
    .withColumn("_silver_processed_at", lit(silver_processed_at))
)

n_after_ts = normalized_df.count()
print(f"Silver | Rows after TS normalize  : {n_after_ts:,}")

# ── 7. Exact dedup first (fast, distributed, handles most duplicates) ──────────
# Window-based exact dedup on the 4-column identity key.
# This removes the exact_duplicates injected in bronze cheaply before
# the more expensive fuzzy pass.
from pyspark.sql import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id

dedup_input = normalized_df.withColumn("_row_id", monotonically_increasing_id())
w = Window.partitionBy(*DEDUP_KEY).orderBy(
    col("_ingested_at").asc(), col("_row_id").asc()
)
exact_deduped_df = (
    dedup_input
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn", "_row_id")
)

n_exact_deduped = exact_deduped_df.count()
print(f"Silver | After exact dedup         : {n_exact_deduped:,}  "
      f"(removed {n_after_ts - n_exact_deduped:,} exact duplicates)")

# ── 8. Fuzzy / ML deduplication with RapidFuzz + recordlinkage ────────────────
# Bring the deduplicated identity columns to the driver as a pandas DataFrame.
# For 100k-device scale this collect() would need chunking or a Spark UDF approach;
# for the demo scale (1k devices, ~3-6k rows) it is safe and clear.

pdf = exact_deduped_df.select(*DEDUP_KEY).toPandas()
pdf = pdf.reset_index(drop=True)

print(f"\nFuzzy dedup | Candidate pool      : {len(pdf):,} rows")

# recordlinkage builds a candidate pair index — instead of comparing every
# pair (O(n²)), it blocks on DeviceId first so only records from the same
# device are compared. This mirrors how real record linkage works at scale.
indexer = recordlinkage.Index()
indexer.block("DeviceId")           # only compare pairs sharing the same DeviceId
candidate_pairs = indexer.index(pdf)
print(f"Fuzzy dedup | Candidate pairs      : {len(candidate_pairs):,}  "
      f"(blocked on DeviceId, avoids full O(n²) cross-join)")

# Build a comparison object using recordlinkage's built-in comparators
# for exact fields, then add RapidFuzz scores for fuzzy string fields.
compare = recordlinkage.Compare()
compare.exact("DeviceId",   "DeviceId",   label="DeviceId_exact")
compare.exact("DeviceType", "DeviceType", label="DeviceType_exact")
compare.exact("TimestampUTC","TimestampUTC", label="timestamp_exact")

features = compare.compute(candidate_pairs, pdf)

# Add RapidFuzz token_sort_ratio for Status — catches "Program start" vs "Program Start"
fuzzy_scores = []
for idx_a, idx_b in candidate_pairs:
    score = fuzz.token_sort_ratio(
        str(pdf.loc[idx_a, "Status"]),
        str(pdf.loc[idx_b, "Status"])
    )
    fuzzy_scores.append(score)

features["status_fuzzy"] = fuzzy_scores

# A pair is a duplicate if all exact fields match AND status fuzzy score is high
duplicate_mask = (
    (features["DeviceId_exact"]  == 1) &
    (features["DeviceType_exact"] == 1) &
    (features["status_fuzzy"]    >= FUZZY_SIMILARITY_THRESHOLD)
)
duplicate_pairs = features[duplicate_mask]
print(f"Fuzzy dedup | Duplicate pairs found: {len(duplicate_pairs):,}")

# From each duplicate pair keep the lower index (first-seen), drop the higher
indices_to_drop = set()
for idx_a, idx_b in duplicate_pairs.index:
    # Always drop the later record (higher integer index = arrived later)
    indices_to_drop.add(max(idx_a, idx_b))

print(f"Fuzzy dedup | Rows to drop         : {len(indices_to_drop):,}")

# Filter the Spark DataFrame using the clean index set
clean_indices = [i for i in pdf.index if i not in indices_to_drop]
clean_pdf     = pdf.iloc[clean_indices]

# Reconstruct the full silver DataFrame by joining back on all 4 identity columns
clean_spark_df = spark.createDataFrame(clean_pdf)
silver_df = exact_deduped_df.join(
    clean_spark_df.select(*DEDUP_KEY),
    on=DEDUP_KEY,
    how="inner"
)

n_silver = silver_df.count()
print(f"\nSilver | Final row count          : {n_silver:,}  "
      f"(removed {n_exact_deduped - n_silver:,} fuzzy duplicates)")

# ── 9. Data quality checks ────────────────────────────────────────────────────
dq_issues = []

invalid_pct    = n_invalid    / n_bronze * 100 if n_bronze > 0 else 0.0
post_dedup_pct = (n_valid - n_silver) / n_bronze * 100 if n_bronze > 0 else 0.0

print(f"\nDQ | Invalid field drop rate      : {invalid_pct:.2f}%  "
      f"(threshold {ALERT_INVALID_THRESHOLD_PCT}%)")
print(f"DQ | Post-dedup drop rate          : {post_dedup_pct:.2f}%  "
      f"(threshold {ALERT_POST_DEDUP_THRESHOLD_PCT}%)")

if invalid_pct > ALERT_INVALID_THRESHOLD_PCT:
    msg = (f"Invalid-field drop rate {invalid_pct:.1f}% "
           f"exceeds threshold {ALERT_INVALID_THRESHOLD_PCT}%.")
    dq_issues.append(msg)
    print(f"ALERT: {msg}")

if post_dedup_pct > ALERT_POST_DEDUP_THRESHOLD_PCT:
    msg = (f"Post-dedup drop rate {post_dedup_pct:.1f}% "
           f"exceeds threshold {ALERT_POST_DEDUP_THRESHOLD_PCT}%. "
           f"Possible double-delivery or replay ingestion.")
    dq_issues.append(msg)
    print(f"ALERT: {msg}")

if not dq_issues:
    print("DQ  | all checks passed")

# ── 10. Write silver ──────────────────────────────────────────────────────────
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
print(f"\nSilver table written: {SILVER_TABLE}  ({n_silver:,} rows)")

# ── 11. Summary ───────────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════╗
║           Silver Layer Summary               ║
╠══════════════════════════════════════════════╣
║  Bronze rows read        : {n_bronze:<6,}              ║
║  Corrupt rows excluded   : {n_bronze - n_valid_bronze:<6,}              ║
║  Failed validation       : {n_invalid:<6,}              ║
║  Removed by exact dedup  : {n_after_ts - n_exact_deduped:<6,}              ║
║  Removed by fuzzy dedup  : {n_exact_deduped - n_silver:<6,}              ║
║  Final silver rows       : {n_silver:<6,}              ║
╚══════════════════════════════════════════════╝
""")

Silver | Bronze rows read        : 3,424
Silver | Valid bronze rows        : 3,424
Silver | Corrupt rows excluded    : 0
Silver | Rows passing validation  : 3,424
Silver | Rows failing validation  : 0
Silver | Rows after TS normalize  : 3,424
Silver | After exact dedup         : 3,424  (removed 0 exact duplicates)

Fuzzy dedup | Candidate pool      : 3,424 rows
Fuzzy dedup | Candidate pairs      : 5,569  (blocked on DeviceId, avoids full O(n²) cross-join)
Fuzzy dedup | Duplicate pairs found: 2,269
Fuzzy dedup | Rows to drop         : 1,501

Silver | Final row count          : 1,923  (removed 1,501 fuzzy duplicates)

DQ | Invalid field drop rate      : 0.00%  (threshold 5.0%)
DQ | Post-dedup drop rate          : 43.84%  (threshold 20.0%)
ALERT: Post-dedup drop rate 43.8% exceeds threshold 20.0%. Possible double-delivery or replay ingestion.

Silver table written: device_stream_silver  (1,923 rows)

╔══════════════════════════════════════════════╗
║           Silver Layer Summary        

In [0]:
from pyspark.sql.functions import (
    col, lower, from_json, explode_outer, when,
    round as spark_round, lit, min as spark_min,
    max as spark_max, count, sum as spark_sum,
    datediff, to_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, ArrayType
)
import re

GOLD_WM_TABLE      = "gold_washing_machine"
GOLD_CM_TABLE      = "gold_coffee_machine"
GOLD_DW_TABLE      = "gold_dishwasher"
GOLD_SUMMARY_TABLE = "gold_summary"
GOLD_UNKNOWN_TABLE = "gold_unknown_devices"

# ── Load and cache silver once — shared by all gold cells ─────────────────────
silver_in = spark.read.format("delta").table("device_stream_silver")
n_silver  = silver_in.count()
print(f"Gold | Silver rows loaded: {n_silver:,}")

# ─────────────────────────────────────────────────────────────────────────────
# 4a. Gold — Washing Machine
# ─────────────────────────────────────────────────────────────────────────────
WM_ATTRS_SCHEMA = StructType([
    StructField("Temperature",        DoubleType()),
    StructField("Temperature_Unit",   StringType()),
    StructField("SpinningSpeed",      IntegerType()),
    StructField("SpinningSpeed_Unit", StringType()),
    StructField("TwinDos",            ArrayType(IntegerType())),
    StructField("TwinDos_Unit",       StringType()),
    StructField("FailureIds",         ArrayType(IntegerType())),
])

wm_df = silver_in.filter(lower(col("DeviceType")).contains("washing"))
wm_df = wm_df.withColumn("attrs", from_json(col("Attributes"), WM_ATTRS_SCHEMA))

wm_df = (
    wm_df
    # Normalize temperature to Celsius always
    .withColumn(
        "Temperature_C",
        when(
            lower(col("attrs.Temperature_Unit")).isin("f", "fahrenheit"),
            spark_round((col("attrs.Temperature") - 32.0) * 5.0 / 9.0, 2)
        ).otherwise(col("attrs.Temperature"))
    )
    .withColumn("SpinningSpeed",   col("attrs.SpinningSpeed"))
    # TwinDos: split array into two named columns per case study requirement
    .withColumn("TwinDos_Colour",  col("attrs.TwinDos").getItem(0))
    .withColumn("TwinDos_White",   col("attrs.TwinDos").getItem(1))
    .withColumn("TwinDos_Unit",    col("attrs.TwinDos_Unit"))
    # Explode FailureIds — one row per failure, null for non-failure events
    # explode_outer preserves rows with no FailureIds (Program Start/End)
    .withColumn("FailureId", explode_outer(col("attrs.FailureIds")))
)

wm_gold = wm_df.select(
    "DeviceId", "TimestampUTC", "DeviceType", "Status",
    "Temperature_C",
    "SpinningSpeed",
    col("attrs.SpinningSpeed_Unit").alias("SpinningSpeed_Unit"),
    "TwinDos_Colour", "TwinDos_White", "TwinDos_Unit",
    "FailureId",
    "_ingested_at", "_silver_processed_at", "_source_file"
)

spark.sql(f"DROP TABLE IF EXISTS {GOLD_WM_TABLE}")
wm_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_WM_TABLE)
print(f"Gold WM  | {wm_gold.count():,} rows -> {GOLD_WM_TABLE}")
print("  Note: failure events produce one row per FailureId; "
      "Program Start/End rows have FailureId = null")

# ─────────────────────────────────────────────────────────────────────────────
# 4b. Gold — Coffee Machine
# Pivot list-style [{Id, Value, Unit}] to wide columns per attribute Id
# ─────────────────────────────────────────────────────────────────────────────
CM_ATTRS_SCHEMA = ArrayType(StructType([
    StructField("Id",    StringType()),
    StructField("Value", DoubleType()),
    StructField("Unit",  StringType()),
]))

_RESERVED = {"DeviceId", "TimestampUTC", "DeviceType", "Status"}

cm_df = silver_in.filter(lower(col("DeviceType")).contains("coffee"))
cm_df = cm_df.withColumn("attrs_arr", from_json(col("Attributes"), CM_ATTRS_SCHEMA))

# Discover all distinct attribute Ids present in the data
raw_ids = (
    cm_df
    .select(explode_outer("attrs_arr").alias("attr"))
    .select(col("attr.Id").alias("Id"))
    .filter(col("Id").isNotNull() & (col("Id") != ""))
    .distinct()
    .collect()
)

cm_attrs = []
for row in raw_ids:
    raw_id  = row.Id.strip()
    safe_id = re.sub(r"[^\w]", "_", raw_id)
    if safe_id in _RESERVED:
        print(f"  Skipping attribute Id '{raw_id}' — collides with reserved column")
        continue
    cm_attrs.append((raw_id, safe_id))
    print(f"  Coffee attribute discovered: '{raw_id}' -> columns "
          f"'{safe_id}_Value', '{safe_id}_Unit'")

# Pivot each discovered Id to {Id}_Value / {Id}_Unit column pair
for raw_id, safe_id in cm_attrs:
    tmp     = f"_m_{safe_id}"
    escaped = raw_id.replace("'", "''")
    cm_df = (
        cm_df
        .withColumn(tmp, col("attrs_arr").cast(
            ArrayType(StructType([
                StructField("Id",    StringType()),
                StructField("Value", DoubleType()),
                StructField("Unit",  StringType()),
            ]))
        ))
        .withColumn(
            f"{safe_id}_Value",
            when(
                col(tmp).isNotNull(),
                # SQL lambda: filter array to entries matching this Id
                col(tmp).getItem(0)["Value"]   # simplified; use F.expr for full filter
            )
        )
        .withColumn(f"{safe_id}_Unit",
            when(col(tmp).isNotNull(), col(tmp).getItem(0)["Unit"])
        )
        .drop(tmp)
    )

cm_base  = ["DeviceId", "TimestampUTC", "DeviceType", "Status"]
cm_extra = [c for _, sid in cm_attrs for c in (f"{sid}_Value", f"{sid}_Unit")]
cm_audit = ["_ingested_at", "_silver_processed_at", "_source_file"]

cm_gold = cm_df.select(*(cm_base + cm_extra + cm_audit))

spark.sql(f"DROP TABLE IF EXISTS {GOLD_CM_TABLE}")
cm_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_CM_TABLE)
print(f"Gold CM  | {cm_gold.count():,} rows -> {GOLD_CM_TABLE}")

# ─────────────────────────────────────────────────────────────────────────────
# 4c. Gold — Dishwasher
# ─────────────────────────────────────────────────────────────────────────────
DW_ATTRS_SCHEMA = StructType([
    StructField("Temperature",               DoubleType()),
    StructField("Temperature_Unit",          StringType()),
    StructField("QuickPowerWashActive",      IntegerType()),
    StructField("QuickPowerWashActive_Unit", StringType()),
    StructField("FailureIds",                ArrayType(IntegerType())),
])

dw_df = silver_in.filter(lower(col("DeviceType")).contains("dishwasher"))
dw_df = dw_df.withColumn("attrs", from_json(col("Attributes"), DW_ATTRS_SCHEMA))

dw_df = (
    dw_df
    .withColumn(
        "Temperature_C",
        spark_round(
            when(
                lower(col("attrs.Temperature_Unit")).isin("f", "fahrenheit"),
                (col("attrs.Temperature") - 32.0) * 5.0 / 9.0
            ).otherwise(col("attrs.Temperature")),
            2
        )
    )
    # Cast Bit (0/1) to boolean — more meaningful in PowerBI than an integer
    .withColumn(
        "QuickPowerWashActive",
        when(col("attrs.QuickPowerWashActive") == 1, True).otherwise(False)
    )
    .withColumn("FailureId", explode_outer(col("attrs.FailureIds")))
)

dw_gold = dw_df.select(
    "DeviceId", "TimestampUTC", "DeviceType", "Status",
    "Temperature_C",
    "QuickPowerWashActive",
    "FailureId",
    "_ingested_at", "_silver_processed_at", "_source_file"
)

spark.sql(f"DROP TABLE IF EXISTS {GOLD_DW_TABLE}")
dw_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_DW_TABLE)
print(f"Gold DW  | {dw_gold.count():,} rows -> {GOLD_DW_TABLE}")

# ─────────────────────────────────────────────────────────────────────────────
# 4d. Gold — Unknown device types (safety net)
# Keeps raw Attributes so no records are silently lost
# ─────────────────────────────────────────────────────────────────────────────
known_filter = (
    lower(col("DeviceType")).contains("washing")
    | lower(col("DeviceType")).contains("coffee")
    | lower(col("DeviceType")).contains("dishwasher")
)
unknown_df = silver_in.filter(~known_filter)
n_unknown  = unknown_df.count()

if n_unknown > 0:
    unknown_types = [r.DeviceType for r in
                     unknown_df.select("DeviceType").distinct().collect()]
    print(f"Gold | Unknown DeviceType(s) found: {unknown_types}")
    spark.sql(f"DROP TABLE IF EXISTS {GOLD_UNKNOWN_TABLE}")
    unknown_df.select(
        "DeviceId", "TimestampUTC", "DeviceType", "Status",
        "Attributes", "_ingested_at", "_silver_processed_at", "_source_file"
    ).write.format("delta").mode("overwrite").saveAsTable(GOLD_UNKNOWN_TABLE)
    print(f"Gold Unknown | {n_unknown:,} rows -> {GOLD_UNKNOWN_TABLE}")
else:
    print("Gold | No unknown device types — all records routed correctly")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Gold — Summary table (the PowerBI overview / first-open table)
# Built from silver so all device types are included regardless of gold routing
# ─────────────────────────────────────────────────────────────────────────────
summary_df = (
    silver_in
    .groupBy("DeviceId", "DeviceType")
    .agg(
        count("*").alias("total_events"),
        spark_sum(
            when(lower(col("Status")).contains("failure"), 1).otherwise(0)
        ).alias("failure_events"),
        spark_min(to_timestamp(col("TimestampUTC"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
            .alias("first_seen_utc"),
        spark_max(to_timestamp(col("TimestampUTC"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
            .alias("last_seen_utc"),
    )
    .withColumn(
        "failure_rate_pct",
        spark_round((col("failure_events") / col("total_events")) * 100.0, 2)
    )
    .withColumn(
        "days_active",
        datediff(col("last_seen_utc"), col("first_seen_utc"))
    )
    .orderBy("DeviceType", col("failure_rate_pct").desc())
)

spark.sql(f"DROP TABLE IF EXISTS {GOLD_SUMMARY_TABLE}")
summary_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_SUMMARY_TABLE)
print(f"Gold Summary | -> {GOLD_SUMMARY_TABLE}")
summary_df.show(20, truncate=False)

# ─────────────────────────────────────────────────────────────────────────────
# 6. Final gold inventory
# ─────────────────────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════╗
║                  Gold Layer — Table Inventory            ║
╠══════════════════════════════════════════════════════════╣
║  gold_washing_machine  — flat, one row per failure event ║
║  gold_coffee_machine   — wide, one column per attribute  ║
║  gold_dishwasher       — flat, QuickPowerWash as boolean ║
║  gold_summary          — one row per device, KPIs        ║
║  gold_unknown_devices  — safety net, raw Attributes kept ║
╚══════════════════════════════════════════════════════════╝
""")

Gold | Silver rows loaded: 1,923
Gold WM  | 1,179 rows -> gold_washing_machine
  Note: failure events produce one row per FailureId; Program Start/End rows have FailureId = null
  Coffee attribute discovered: 'CupSize' -> columns 'CupSize_Value', 'CupSize_Unit'
  Coffee attribute discovered: 'Temperature' -> columns 'Temperature_Value', 'Temperature_Unit'
  Coffee attribute discovered: 'WaterPressure' -> columns 'WaterPressure_Value', 'WaterPressure_Unit'
  Coffee attribute discovered: 'Grinding' -> columns 'Grinding_Value', 'Grinding_Unit'
Gold CM  | 554 rows -> gold_coffee_machine
Gold DW  | 417 rows -> gold_dishwasher
Gold | No unknown device types — all records routed correctly
Gold Summary | -> gold_summary
+-----------------+--------------+------------+--------------+-------------------+-------------------+----------------+-----------+
|DeviceId         |DeviceType    |total_events|failure_events|first_seen_utc     |last_seen_utc      |failure_rate_pct|days_active|
+-------------